# Uncertainty Sampling for Expert Review

This notebook scores chatbot responses using entropy and logit-margin heuristics so clinicians review the most uncertain generations first. It creates a prioritised JSONL file that feeds the annotation UI.

In [41]:
import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

BASE_DIR = Path.cwd()
MODEL_OUTPUT_DIR = BASE_DIR / "model_output"
FEEDBACK_DIR = BASE_DIR / "expert_feedback"
FEEDBACK_DIR.mkdir(exist_ok=True)

SOURCE_CANDIDATES = [
    MODEL_OUTPUT_DIR / "predictions_with_logits.jsonl",
    MODEL_OUTPUT_DIR / "predictions.jsonl",
    MODEL_OUTPUT_DIR / "predictions.json",
    MODEL_OUTPUT_DIR / "eval_predictions.jsonl",
    MODEL_OUTPUT_DIR / "eval_predictions.json",
    MODEL_OUTPUT_DIR / "generated_responses.jsonl",
]

PRIORITISED_PATH = FEEDBACK_DIR / "pending_samples.jsonl"
LOW_CONFIDENCE_EXPORT = FEEDBACK_DIR / "low_confidence_samples.jsonl"
ENRICHED_EXPORT = MODEL_OUTPUT_DIR / "predictions_with_uncertainty.jsonl"
DEFAULT_TOP_K = 200


In [42]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def read_json(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ("predictions", "responses", "items", "samples"):
            if key in data and isinstance(data[key], list):
                return data[key]
    return []


def write_jsonl(path: Path, records: Iterable[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def softmax(logits: Sequence[float]) -> np.ndarray:
    logits = np.array(logits, dtype=np.float64)
    logits = logits - logits.max(initial=0.0)
    exps = np.exp(logits)
    denom = exps.sum()
    if denom == 0:
        return np.full_like(exps, 1.0 / len(exps))
    return exps / denom


def extract_token_probabilities(sample: Dict[str, Any]) -> Optional[List[float]]:
    if "token_probabilities" in sample:
        return [float(p) for p in sample["token_probabilities"] if p is not None]
    if "token_logprobs" in sample:
        return [math.exp(float(lp)) for lp in sample["token_logprobs"] if lp is not None]
    if "token_scores" in sample:
        return [float(p) for p in sample["token_scores"] if p is not None]
    return None


def extract_token_logits(sample: Dict[str, Any]) -> Optional[List[Sequence[float]]]:
    if "token_logits" in sample and sample["token_logits"]:
        return sample["token_logits"]
    if "logits" in sample and isinstance(sample["logits"], list):
        return sample["logits"]
    return None


def normalise_probabilities(probs: Sequence[float]) -> np.ndarray:
    probs = np.asarray(probs, dtype=np.float64)
    probs = np.clip(probs, 1e-9, 1.0)
    total = probs.sum()
    if total == 0:
        return np.full_like(probs, 1.0 / len(probs))
    return probs / total


def token_entropy(probs: Sequence[float]) -> float:
    probs = normalise_probabilities(probs)
    return float(-np.sum(probs * np.log(probs)))


def entropy_stats(token_probs: Sequence[Sequence[float]]) -> Tuple[float, float]:
    entropies: List[float] = []
    maxima: List[float] = []
    for prob in token_probs:
        if not isinstance(prob, (list, tuple, np.ndarray)) or len(prob) == 0:
            continue
        entropies.append(token_entropy(prob))
        maxima.append(math.log(len(prob)))
    if not entropies:
        return float("nan"), float("nan")
    return float(np.mean(entropies)), float(np.mean(maxima))


def logit_margin(token_probs: Sequence[Sequence[float]]) -> float:
    margins: List[float] = []
    for probs in token_probs:
        if not isinstance(probs, (list, tuple, np.ndarray)) or len(probs) < 2:
            continue
        arr = normalise_probabilities(probs)
        top2 = np.sort(arr)[-2:]
        if len(top2) == 2:
            margins.append(float(top2[-1] - top2[-2]))
    if not margins:
        return float("nan")
    return float(np.mean(margins))


@dataclass
class UncertaintyResult:
    sequence_entropy: Optional[float]
    max_entropy: Optional[float]
    logit_margin: Optional[float]
    combined_score: float


def combine_scores(
    entropy_value: Optional[float],
    max_entropy: Optional[float],
    margin_value: Optional[float],
    entropy_weight: float = 0.7,
) -> UncertaintyResult:
    entropy_score = 0.0
    if entropy_value is not None and max_entropy is not None and max_entropy > 0 and not math.isnan(entropy_value):
        entropy_score = min(max(entropy_value / max_entropy, 0.0), 1.0)

    margin_score = 0.0
    if margin_value is not None and not math.isnan(margin_value):
        margin_score = 1.0 - min(max(margin_value, 0.0), 1.0)

    combined = entropy_weight * entropy_score + (1 - entropy_weight) * margin_score
    return UncertaintyResult(
        sequence_entropy=entropy_value if entropy_value is not None and not math.isnan(entropy_value) else None,
        max_entropy=max_entropy if max_entropy is not None and not math.isnan(max_entropy) else None,
        logit_margin=margin_value if margin_value is not None and not math.isnan(margin_value) else None,
        combined_score=float(combined),
    )


In [43]:
def build_token_distributions(sample: Dict[str, Any]) -> List[np.ndarray]:
    distributions: List[np.ndarray] = []

    token_logits = extract_token_logits(sample)
    if token_logits:
        for token in token_logits:
            if isinstance(token, dict):
                values = token.get("values") or token.get("logits") or token.get("topk")
                if values:
                    distributions.append(softmax(values))
            elif isinstance(token, (list, tuple, np.ndarray)):
                distributions.append(softmax(token))
        if distributions:
            return distributions

    confidences = sample.get("token_confidences")
    if isinstance(confidences, list):
        for item in confidences:
            if isinstance(item, dict) and "prob" in item:
                p = float(np.clip(item["prob"], 1e-6, 1 - 1e-6))
                distributions.append(np.array([p, 1 - p]))
        if distributions:
            return distributions

    token_probs = extract_token_probabilities(sample)
    if token_probs:
        for prob in token_probs:
            p = float(np.clip(prob, 1e-6, 1 - 1e-6))
            distributions.append(np.array([p, 1 - p]))
    return distributions


def compute_uncertainty(sample: Dict[str, Any], entropy_weight: float = 0.7) -> UncertaintyResult:
    token_distributions = build_token_distributions(sample)
    if token_distributions:
        entropy_val, max_entropy = entropy_stats(token_distributions)
        margin_val = logit_margin(token_distributions)
    else:
        entropy_val, max_entropy, margin_val = None, None, None
    return combine_scores(entropy_val, max_entropy, margin_val, entropy_weight=entropy_weight)


In [44]:
def load_predictions(source_candidates: Sequence[Path]) -> List[Dict[str, Any]]:
    for path in source_candidates:
        if path.suffix == ".jsonl":
            records = read_jsonl(path)
        else:
            records = read_json(path)
        if records:
            print(f"Loaded {len(records)} samples from {path}")
            return records
    raise FileNotFoundError("No prediction file found. Export model outputs (*.json or *.jsonl) with token-level confidences.")


def enrich_with_uncertainty(records: List[Dict[str, Any]], entropy_weight: float = 0.7) -> List[Dict[str, Any]]:
    enriched: List[Dict[str, Any]] = []
    for idx, record in enumerate(records):
        scores = compute_uncertainty(record, entropy_weight=entropy_weight)
        annotated = {
            **record,
            "id": record.get("id") or record.get("sample_id") or record.get("uuid") or str(idx),
            "uncertainty": {
                "sequence_entropy": scores.sequence_entropy,
                "max_entropy": scores.max_entropy,
                "logit_margin": scores.logit_margin,
                "combined": float(scores.combined_score),
            },
        }
        annotated["uncertainty_score"] = annotated["uncertainty"]["combined"]
        enriched.append(annotated)
    return enriched


def prioritise_samples(records: List[Dict[str, Any]], top_k: int = DEFAULT_TOP_K) -> List[Dict[str, Any]]:
    scored: List[Dict[str, Any]] = []
    fallback: List[Dict[str, Any]] = []

    for record in records:
        score = record.get("uncertainty_score")
        if isinstance(score, (int, float)) and not math.isnan(score):
            scored.append(record)
        else:
            fallback.append(record)

    ranked = sorted(scored, key=lambda r: r["uncertainty_score"], reverse=True)
    if top_k:
        ranked = ranked[:top_k]

    if len(ranked) < top_k and fallback:
        ranked.extend(fallback[: max(0, top_k - len(ranked))])

    return ranked


In [45]:
try:
    raw_predictions = load_predictions(SOURCE_CANDIDATES)
except FileNotFoundError:
    fallback_path = BASE_DIR / "processed_data" / "processed_dev.json"
    if fallback_path.exists():
        print("Falling back to processed_dev.json (no confidence scores detected).")
        raw_predictions = read_json(fallback_path)
    else:
        raise RuntimeError(
            "No predictions found. Run eval_model.ipynb or export chatbot generations with token probabilities."
        )

scored_predictions = enrich_with_uncertainty(raw_predictions)
ranked_for_review = prioritise_samples(scored_predictions, top_k=DEFAULT_TOP_K)

write_jsonl(ENRICHED_EXPORT, scored_predictions)
write_jsonl(LOW_CONFIDENCE_EXPORT, ranked_for_review)
write_jsonl(PRIORITISED_PATH, ranked_for_review)

missing_confidence = sum(
    1 for record in scored_predictions if record["uncertainty"].get("sequence_entropy") is None
)
print(f"Saved enriched predictions → {ENRICHED_EXPORT.relative_to(BASE_DIR)}")
print(f"Saved top-{len(ranked_for_review)} low-confidence responses → {LOW_CONFIDENCE_EXPORT.relative_to(BASE_DIR)}")
print(f"Updated annotation queue seed file → {PRIORITISED_PATH.relative_to(BASE_DIR)}")
if missing_confidence:
    print(
        f"Warning: {missing_confidence} samples lacked token-level confidence data. "
        "Consider enabling `output_scores=True` when generating responses."
    )


Falling back to processed_dev.json (no confidence scores detected).
Saved enriched predictions → model_output/predictions_with_uncertainty.jsonl
Saved top-60 low-confidence responses → expert_feedback/low_confidence_samples.jsonl
Updated annotation queue seed file → expert_feedback/pending_samples.jsonl


### Next steps

- Re-run this notebook after every evaluation run to refresh `expert_feedback/pending_samples.jsonl`.
- Inspect `model_output/predictions_with_uncertainty.jsonl` to audit the scores.
- The annotation UI notebook will automatically pick up the prioritised queue the next time you launch it.
- If your decoder cannot emit token-level logits, consider enabling `output_scores=True` and `return_dict_in_generate=True` in the generation loop to populate the required fields.